# 00 — Ground truth, and the standard results

Nothing about the machine learning is believable until the ground truth reproduces what the
system is known to do. This notebook is that check, plus the frozen datasets everything
downstream inherits.

Two kinds of row below and they are not the same evidence. C±, ∇·f and ρ_H are **closed forms
evaluated** — they check the formulas were transcribed right and cannot fail otherwise. The
measured divergence, the spectrum, its sum against ∇·f, Kaplan–Yorke, the symmetry residual
and the Lorenz map's slope are **numerical and can fail**; those are the integrator check.

Everything here is produced by `run/ground_truth.py`; this reads its output.

In [1]:
import sys, json, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks'
                       else pathlib.Path.cwd()))
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt, torch
from l63 import ARTIFACTS, evaluate as E, plots as P
from l63.data import make_datasets
from run.report import clean, load_model, load_rows, summarise

gt = json.load(open(ARTIFACTS / 'ground_truth.json'))
S  = clean(summarise(load_rows()))
DATA = list(gt['datasets'])                      # 'ode', 'sde', 'sde015'

def num(v, w=6, p=2):
    """A ruler value, or an em dash where it is genuinely undefined."""
    if isinstance(v, dict):
        v = v.get('median')
    undefined = v is None or v != v          # None from clean(), bare NaN from the json
    return f'{"—":>{w}}' if undefined else f'{v:{w}.{p}f}'

def rng(v, p=2):
    if v is None or v.get('lo') is None or v['lo'] != v['lo']:
        return '—'
    return f"[{v['lo']:.{p}f}–{v['hi']:.{p}f}]"

print(len(S), 'model x dataset entries ·', len(DATA), 'datasets')

43 model x dataset entries · 3 datasets


## Numbers

In [2]:
k = gt['known']
print("--- closed forms, transcription checks (cannot fail) ---")
print(f"C+                {k['C_plus']}          expected (8.485, 8.485, 27)")
print(f"div f (formula)   {k['divergence']:.4f}                expected -13.6667")
print(f"rho_H             {k['rho_hopf']:.4f}                 expected 24.7368")
print("--- numerical, these are the integrator check ---")
print(f"div f (measured)  {k['divergence_measured']:.6f}   max deviation {k['divergence_measured_max_dev']:.1e}")
print(f"symmetry residual {k['symmetry_residual']:.1e}                  expected 0")
print(f"spectrum          {[round(v,4) for v in k['spectrum']]}")
print(f"  sum             {k['spectrum_sum']:.4f}   vs div f {k['divergence']:.4f}   <- an identity")
print(f"Kaplan-Yorke      {k['kaplan_yorke']:.4f}                 Strogatz measured ~2.05")
print(f"Lorenz map |f'|   min {k['lorenz_map_min_slope']:.3f} over {k['lorenz_map_peaks']} maxima   must be > 1")
print(f"\nlambda_1 of the true map at dt: {gt['lambda_true']:.4f} +- {gt['lambda_true_sd']:.4f}")

for key in DATA:
    r = gt['datasets'][key]
    w = f"   conditional width {r['conditional_width_pct']:.1f}% of scale" if r['kind'] == 'sde' else ''
    print(f"\n{key:7s} b={r['b']:<5g} scale {r['attractor_scale']:.2f}   "
          f"same-cost solver usable {r['euler_bar_steps']} steps   "
          f"ground truth usable {r['floor_steps']} steps{w}")

--- closed forms, transcription checks (cannot fail) ---
C+                [8.485280990600586, 8.485280990600586, 27.0]          expected (8.485, 8.485, 27)
div f (formula)   -13.6667                expected -13.6667
rho_H             24.7368                 expected 24.7368
--- numerical, these are the integrator check ---
div f (measured)  -13.666668   max deviation 0.0e+00
symmetry residual 0.0e+00                  expected 0
spectrum          [0.9091, -0.0061, -14.5786]
  sum             -13.6755   vs div f -13.6667   <- an identity
Kaplan-Yorke      2.0619                 Strogatz measured ~2.05
Lorenz map |f'|   min 1.154 over 534 maxima   must be > 1

lambda_1 of the true map at dt: 0.8974 +- 0.0233

ode     b=0     scale 13.98   same-cost solver usable 32 steps   ground truth usable 362 steps

sde     b=0.6   scale 15.83   same-cost solver usable 26 steps   ground truth usable 23 steps   conditional width 16.5% of scale

sde015  b=0.15  scale 14.41   same-cost solver usable 29 

## Findings

_Written after reading the numbers above._

- 
- 
- 